In [ ]:
"""
Script de automatización: Reporte Diario de Portafolio
Ejecuta cada día y genera un reporte con el estado de las inversiones
"""

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

class AnalizadorPortafolio:
    """Clase para analizar y reportar portafolio de acciones"""

    def __init__(self, tickers, cantidades):
        """
        Inicializar analizador

        Args:
            tickers (list): Lista de símbolos ['AAPL', 'GOOGL', etc.]
            cantidades (list): Número de acciones de cada una [10, 5, etc.]
        """
        self.tickers = tickers
        self.cantidades = cantidades
        self.datos = None

    def obtener_datos(self, periodo='1mo'):
        """Obtener datos históricos de todas las acciones"""
        datos_lista = []

        for ticker, cantidad in zip(self.tickers, self.cantidades):
            accion = yf.Ticker(ticker)
            hist = accion.history(period=periodo)
            info = accion.info

            precio_actual = hist['Close'].iloc[-1]
            precio_anterior = hist['Close'].iloc[-2]
            cambio_diario = ((precio_actual - precio_anterior) / precio_anterior) * 100

            datos_lista.append({
                'Ticker': ticker,
                'Cantidad': cantidad,
                'Precio': precio_actual,
                'Valor Total': precio_actual * cantidad,
                'Cambio %': cambio_diario,
                'Ganancia/Pérdida $': (precio_actual - precio_anterior) * cantidad
            })

        self.datos = pd.DataFrame(datos_lista)
        return self.datos

    def generar_resumen(self):
        """Generar resumen estadístico del portafolio"""
        valor_total = self.datos['Valor Total'].sum()
        cambio_total = self.datos['Ganancia/Pérdida $'].sum()
        cambio_porcentaje = (cambio_total / (valor_total - cambio_total)) * 100

        resumen = f"""
📊 REPORTE DIARIO DE PORTAFOLIO - {datetime.now().strftime('%Y-%m-%d')}
{'=' * 60}

💰 Valor Total del Portafolio: ${valor_total:,.2f}
📈 Cambio del Día: ${cambio_total:,.2f} ({cambio_porcentaje:+.2f}%)

{'MEJOR RENDIMIENTO' if cambio_total > 0 else 'PEOR RENDIMIENTO'} DEL DÍA:
{self.datos.nlargest(1, 'Cambio %')[['Ticker', 'Cambio %', 'Ganancia/Pérdida $']].to_string(index=False)}

DESGLOSE POR ACCIÓN:
{self.datos.to_string(index=False)}
        """
        return resumen

    def crear_grafico(self, filename='portafolio.png'):
        """Crear gráfico visual del portafolio"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

        # Gráfico de distribución de valor
        ax1.pie(self.datos['Valor Total'], labels=self.datos['Ticker'], 
                autopct='%1.1f%%', startangle=90)
        ax1.set_title('Distribución del Portafolio')

        # Gráfico de rendimiento diario
        colors = ['green' if x > 0 else 'red' for x in self.datos['Cambio %']]
        ax2.barh(self.datos['Ticker'], self.datos['Cambio %'], color=colors)
        ax2.set_xlabel('Cambio %')
        ax2.set_title('Rendimiento Diario')
        ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()

        return filename

    def guardar_historial(self, filename='historial_portafolio.csv'):
        """Guardar datos en CSV para análisis histórico"""
        self.datos['Fecha'] = datetime.now().strftime('%Y-%m-%d')

        # Append to CSV (crear si no existe)
        try:
            df_existente = pd.read_csv(filename)
            df_nuevo = pd.concat([df_existente, self.datos], ignore_index=True)
            df_nuevo.to_csv(filename, index=False)
        except FileNotFoundError:
            self.datos.to_csv(filename, index=False)

        print(f"✅ Historial guardado en {filename}")

# EJECUTAR EL SCRIPT
if __name__ == "__main__":
    # Definir portafolio
    mis_tickers = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']
    mis_cantidades = [50, 20, 30, 15, 25]

    # Crear analizador
    analizador = AnalizadorPortafolio(mis_tickers, mis_cantidades)

    # Obtener datos actuales
    print("🔄 Obteniendo datos del mercado...")
    analizador.obtener_datos()

    # Generar y mostrar resumen
    resumen = analizador.generar_resumen()
    print(resumen)

    # Crear visualización
    print("\n📊 Generando gráficos...")
    archivo_grafico = analizador.crear_grafico()
    print(f"✅ Gráfico guardado: {archivo_grafico}")

    # Guardar historial
    analizador.guardar_historial()

    print("\n✅ Reporte completado exitosamente!")